# Box Office Forecasting: the decision log

Every modelling choice in this project, in the order it was made, with the
evidence that settled it. Each section states a decision, shows the check that
informed it, and records what was rejected and why.

The thing being predicted is what a film earns **before it opens**, from what is
knowable on the day the marketing campaign starts.

Run top to bottom. Every number below is recomputed, not transcribed.


In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'boxoffice' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))

import numpy as np, pandas as pd
pd.set_option('display.width', 110)
print('project root:', ROOT)

project root: /Users/thomaslappas/Desktop/MovieGame


---
## Decision 1: what counts as leakage

The obvious rule is *do not use the revenue column*. That is not the rule that
matters. The Clean Sweep seed this project sits beside carries eight columns
that all look like ordinary film metadata and all accumulate **after** a film
has been in front of an audience.

Using any of them produces an excellent score and a worthless model, so they
are named and blocked rather than avoided by discipline.


In [2]:
from boxoffice.model.leakage import BANNED, assert_clean, LeakageError

for col, why in list(BANNED.items())[:9]:
    print(f'{col:22s} {why}')

imdb_rating            accumulates from votes cast after release
imdb_votes             accumulates from votes cast after release
rt_critic              aggregate settles after the review embargo lifts
rt_audience            audience score requires an audience
metascore              aggregate settles after the review embargo lifts
nominations            awarded months to years after release
wins                   awarded months to years after release
award_points           derived from nominations and wins
award_standing         derived from nominations and wins


In [3]:
# The guard fires on a matrix that contains one.
bad = pd.DataFrame({'log_budget': [1.0], 'imdb_rating': [7.2]})
try:
    assert_clean(bad)
except LeakageError as exc:
    print('blocked ->', exc)

blocked -> post-release columns in the feature matrix -> imdb_rating: accumulates from votes cast after release


---
## Decision 2: the leak that survives code review

A director's prior average gross is a legitimate feature. The same average
computed over their **whole career**, including films that had not opened yet,
is not, and the column name is identical either way. A plain `groupby` gives
you the second one.

`AsOf` sorts by release date and lets each row see only strictly earlier rows.
The case worth testing is two films opening the same day: they must both see
the history as of that morning, and neither may see the other.


In [4]:
from boxoffice.model.leakage import AsOf

demo = pd.DataFrame({
    'title':        ['A', 'B', 'C', 'D'],
    'director_id':  ['d1'] * 4,
    'release_date': pd.to_datetime(['2010-01-01','2012-01-01','2014-06-01','2014-06-01']),
    'log_ww':       [10.0, 20.0, 99.0, 99.0],
})
demo['prior_median'] = AsOf('director_id', 'log_ww').transform(demo, 'median')
demo

,title,director_id,release_date,log_ww,prior_median
0,A,d1,2010-01-01,10.0,NaN
1,B,d1,2012-01-01,20.0,10.0
2,C,d1,2014-06-01,99.0,15.0
3,D,d1,2014-06-01,99.0,15.0


In [5]:
prior = demo['prior_median']
checks = {
    'first film has no history':      pd.isna(prior.iloc[0]),
    'B sees only A':                  prior.iloc[1] == 10.0,
    'C sees A and B':                 prior.iloc[2] == 15.0,
    'D, same day as C, sees the same': prior.iloc[3] == 15.0,
    'neither sees the other s 99':    99.0 not in set(prior.dropna()),
}
for name, ok in checks.items():
    print(f'{"PASS" if ok else "FAIL"}  {name}')

PASS  first film has no history
PASS  B sees only A
PASS  C sees A and B
PASS  D, same day as C, sees the same
PASS  neither sees the other s 99


---
## Decision 3: the sampling frame

The population is films that were financed and given a theatrical release, so
the frame has to be defined by pre-release facts only.

TMDB tags almost everything theatrical and returns **37,486 titles for 2019
alone**, nearly all festival and direct-to-video tail with no budget and no
gross. Every convenient way to cut that down selects on the outcome:

| Filter | Verdict |
|---|---|
| Sort by revenue | rejected: selects on the target |
| Filter by vote count | rejected: selects on who eventually saw it |
| Filter by popularity | rejected: the same, measured today |
| **Filter by distributor** | **accepted: known months before release** |

A distributor is attached long before opening weekend, so anchoring on one
keeps the sample definition itself free of the answer.


In [6]:
from boxoffice.pipeline.fetch import DISTRIBUTORS, BUDGET_FLOOR

feat = pd.read_parquet(ROOT / 'backend/boxoffice/data/features.parquet')
print(f'{len(DISTRIBUTORS)} distributors in the frame, budget floor ${BUDGET_FLOOR:,}')
print(f'{len(feat):,} films, {feat.release_date.dt.year.min()}-{feat.release_date.dt.year.max()}')
feat.release_date.dt.year.value_counts().sort_index().tail(8).to_frame('films')

43 distributors in the frame, budget floor $1,000,000
2,577 films, 2000-2029


,films
release_date,
2022,70
2023,76
2024,64
2025,75
2026,33
2027,56
2028,30
2029,1


---
## Decision 4: what a film is, as thirty-five numbers

Three kinds of column, and the third is where the work is.

**What the film is.** Budget, runtime, original language, sequel flag, position
in its franchise, ten genre indicators. All of it printed in the trade press
months out.

**When it opens.** Month, week, and two flags for the corridors that behave
differently: summer and the holiday run. A date is not an ordinal quantity, so
week number is offered alongside the flags rather than instead of them.

**Who is attached, encoded as history.** This is the part that had to be
designed rather than collected. The obvious move is a categorical column for
the director and another for the studio. On 2,457 films with 1,396
distinct directors, one-hot encoding builds a lookup table: it will memorise the
training years and has nothing at all to say about a director it has not seen.

So nobody enters the matrix as an identity. Each person enters as **their track
record as of this film's release date** -- prior median log gross, prior maximum,
and the count of prior films behind those two numbers. Applied to director,
cast, cinematographer, composer and the distributing companies.

Median and maximum are not redundant. A director with a steady $80M career is a
different proposition from one with four flops and a billion-dollar hit, and the
median alone cannot tell them apart. The count is there so the model can
discount a median computed from a single film, which it does.

The payoff is that an unseen name is a *missing value* rather than an unknown
category, which is the same shape as a debut, which is what it actually is.

In [7]:
from boxoffice.model.ablate import GROUPS, BASE

# Coverage on the released films only. The unreleased slate has no budget by
# definition, and counting it here would understate what the model trains on.
released = feat[~feat['is_upcoming'].fillna(False).astype(bool)]

rows = []
for name, cols in [('budget', BASE)] + list(GROUPS.items()):
    rows.append({'group': name, 'n_features': len(cols),
                 'coverage': released[cols].notna().all(axis=1).mean()})
pd.DataFrame(rows).assign(coverage=lambda d: d.coverage.map('{:.0%}'.format))

,group,n_features,coverage
0,budget,1,100%
1,form,4,100%
2,genre,10,100%
3,calendar,4,100%
4,studio,4,98%
5,director,3,50%
6,cast,3,95%
7,cinematographer,3,67%
8,composer,3,74%


---
## Decision 5: no imputation

Half the sample has a director with no earlier film in it. That is not missing
data to be filled; it is a fact about the film. Imputing the median would tell
the model every debut is average.

`HistGradientBoostingRegressor` splits on missingness directly, so *unknown*
stays a category rather than becoming a guess.


In [8]:
cols = [c for c in feat.columns if not c.startswith('y_')
        and c not in ('film_key','title','imdb_id','release_date')]
cov = feat[cols].notna().mean().sort_values()
cov.head(8).to_frame('coverage').style.format('{:.0%}') if hasattr(cov, 'style') else cov.head(8)

director_prior_median_log           0.502910
director_prior_max_log              0.502910
cinematographer_prior_max_log       0.663174
cinematographer_prior_median_log    0.663174
composer_prior_max_log              0.711292
composer_prior_median_log           0.711292
composer_prior_count                0.913077
cinematographer_prior_count         0.940241
dtype: float64

---
## Decision 6: the bar to clear

Not R-squared, and not the median film. Box office spans five orders of
magnitude, so squared error in dollars is decided by half a dozen titles, and
the median is a bar nobody would accept.

**Budget alone is the real baseline**: what a studio spends is most of what a
studio makes. The headline metric is the share of forecasts landing within a
factor of two, because that is how a forecast gets used.

Folds are strictly temporal. Random K-fold would let the model learn 2019 from
2021, which is the second most common way these models get flattered.


In [9]:
from boxoffice.model.train import run, summarise, feature_columns

folds = run(feat, feature_columns(feat))
print(f'{len(folds)} rolling-origin folds, {folds[0].year}-{folds[-1].year}')
summarise(folds)

16 rolling-origin folds, 2010-2025


,estimator,mae_log,within_2x
0,median,1.477540,0.310021
1,budget_only,1.052722,0.473414
2,model,0.950453,0.556384


In [10]:
beat = sum(f.within_2x['model'] > f.within_2x['budget_only'] for f in folds)
print(f'model beats budget-only in {beat} of {len(folds)} folds')
pd.DataFrame([{'year': f.year, 'n': f.n_test,
               'model': f.within_2x['model'],
               'budget_only': f.within_2x['budget_only']} for f in folds])

model beats budget-only in 16 of 16 folds


,year,n,model,budget_only
0,2010,104,0.663462,0.605769
1,2011,105,0.619048,0.590476
2,2012,90,0.600000,0.466667
3,2013,96,0.625000,0.593750
4,2014,99,0.595960,0.525253
5,2015,106,0.575472,0.452830
6,2016,116,0.612069,0.525862
7,2017,100,0.590000,0.510000
8,2018,114,0.543860,0.482456
9,2019,63,0.428571,0.238095


---
## Decision 7: which learner, and what the tuning bought

Gradient boosting was asserted in a module docstring and never justified
against an alternative, which is exactly the kind of claim a reviewer should
distrust. Four candidates on the same folds:

**Ridge** is the honest null. Log gross against log budget is close to linear,
and if a regularised linear model keeps up then the problem does not need
anything clever.

**Random forest** is trees without boosting. If bagging matches, the sequential
fitting is not earning its complexity.

**Boosting out of the box** separates "the family was right" from "the
hyperparameters were right". Quoting a tuned model against untuned alternatives
is a rigged comparison, and it is the usual one.

Ridge and the forest cannot take a missing value, so both get median imputation
**with indicator columns**, which is the strongest form of the thing rather than
the weakest. Beating a straw man here would prove nothing.

In [11]:
from boxoffice.model.bakeoff import summarise as summarise_learners

bake = pd.read_csv(ROOT / 'backend/boxoffice/data/bakeoff.csv')
summarise_learners(bake)

,learner,mae_log,within_2x,folds_beating_shipped
0,budget only,1.052722,0.473414,1/16
1,ridge,0.928095,0.540860,6/16
2,random forest,0.922122,0.556247,8/16
3,boosting (defaults),0.938311,0.550931,7/16
4,boosting (shipped),0.947880,0.557341,-


### What the bakeoff says, including the part that is inconvenient

**The learner is worth a point or two. The features are worth eight.** Every
model given all thirty-five features lands between 54% and 56%; budget alone
sits at 47%. That gap is the whole result, and no choice of estimator moves it.

**Random forest is not worse.** It ties the shipped model on the headline
metric, wins 8 of the 16 folds on it, and carries the lower error in 13. Its
error advantage averages 0.026 fold to fold against a fold-to-fold spread of
0.040, so the honest reading is a tie with a hint, not a winner.

**Tuning bought almost nothing**, and what it bought it paid for: the default
configuration has the lower error and the shipped one has the better within-2x.
That is the tuning doing what it was pointed at, and it is a fraction of a
point.

The booster stays, on a tiebreak that is a stated principle rather than a
number: it takes missing values natively, so it needs no imputation step, and
Decision 5 is the reason that matters. Choosing the model that requires
inventing careers for debut directors, to gain nothing measurable, would
contradict the thing this project is arguing.

---
## Decision 8: which ablation answers the question

The first design added each feature group to a budget-only model. Almost every
group came back *harmful* -- yet all 35 features together gained eleven points.

Two things cause that and only one is interesting. The real one is that these
features only mean something in combination: a star's prior gross says nothing
until you know the budget tier and genre attached to it. The other is a
confound I introduced by holding the boosting configuration fixed across very
different feature counts, so five features and four hundred iterations overfit
in a way thirty-five do not.

Adding a group to a one-feature baseline tests whether it can carry a model
alone. Nobody asked that. **Removing it from the working model** tests what it
contributes alongside the others, which is how it will be used.


In [12]:
abl = pd.read_csv(ROOT / 'backend/boxoffice/data/ablation.csv')
add = abl[abl.design == 'add one to budget'][['group','d_mae','d_2x']]
loo = abl[abl.design == 'remove one from full'][['group','d_mae','d_2x']]
print('ADD ONE TO BUDGET (negative d_mae = helps)'); display(add)
print('REMOVE ONE FROM FULL (positive d_mae = it was carrying weight)'); display(loo)

ADD ONE TO BUDGET (negative d_mae = helps)


,group,d_mae,d_2x
0,budget only (baseline),0.000000,0.000000
1,+ form,-0.054452,0.061872
2,+ genre,0.018042,-0.005157
3,+ calendar,0.046049,-0.004407
4,+ studio,0.000980,0.017321
5,+ director,0.011330,0.011937
6,+ cast,0.075605,-0.010399
7,+ cinematographer,0.032307,-0.004044
8,+ composer,0.036760,0.003575
9,all groups,-0.121696,0.110010


REMOVE ONE FROM FULL (positive d_mae = it was carrying weight)


,group,d_mae,d_2x
10,- form,0.088087,-0.067017
11,- genre,0.013035,-0.007256
12,- calendar,0.007841,-0.015117
13,- studio,0.040276,-0.030974
14,- director,-0.006834,0.004231
15,- cast,-0.000884,-0.009140
16,- cinematographer,0.000047,0.003327
17,- composer,0.005581,-0.017347


### What the second design says

Ranked by what the full model loses when the group is removed:

* **Form** (sequel, franchise position, runtime, language) is the largest single
  contributor, at roughly twice the studio.
* **Studio** is second. The distributor carries more than any of the people.
* **Cinematographer lands at exactly zero.** That was the prediction before the
  run, and it is now the data's answer rather than an assertion.
* **Cast contributes nothing measurable**, and removing the **director** very
  slightly improves the model.

What survives is what the film *is*, not who made it.


---
## Decision 9: prestige, tested and rejected

A film's own Oscar nominations are announced roughly fourteen months after it
opens and are in `BANNED`. The record of the *people attached* is public on
release day, and "from the director of a Best Picture winner" is a line studios
put on posters because it is believed to move tickets. That is a real
hypothesis and it deserved a measurement rather than an opinion.

Thirteen features: Oscar nominations and wins for director, cast and writer as
of the release date, joined through IMDb's `title.principals` to the Academy
data. Alongside them, three other pre-release signals worth the same test: how
crowded the release corridor is, the gap since the last franchise entry, and
the director's most recent gross rather than their career median.

The timing gate is the part that makes it a fair test. A nomination for film
year Y is announced at a ceremony in the first quarter of Y+1, so it is treated
as public from 1 March of Y+1 and counted only for films that opened after that
date. A film opening in February 2016 does not know about the ceremony later
that month.

In [13]:
pres = pd.read_csv(ROOT / 'backend/boxoffice/data/prestige_ablation.csv')
pres[['group', 'n_features', 'mae_log', 'within_2x', 'd_mae', 'd_2x']]

,group,n_features,mae_log,within_2x,d_mae,d_2x
0,full v2 model,48,0.941401,0.557857,0.000000,0.000000
1,all thirteen removed,35,0.947880,0.557341,0.006479,-0.000516
2,- oscar pedigree,38,0.946441,0.555939,0.005040,-0.001918
3,- release competition,47,0.943362,0.555558,0.001962,-0.002300
4,- franchise gap,47,0.943534,0.554620,0.002133,-0.003237
5,- director recency,47,0.942493,0.551913,0.001092,-0.005945


### What thirteen features bought

**Six thousandths of a MAE point, and a tenth of a point of accuracy in the
wrong direction.** Every group sits inside noise under leave-one-out, and the
whole block removed at once changes the model less than re-running it would.

The instability is itself the evidence. An earlier run of this same ablation,
before the feature matrix was rebuilt, had the Oscar block coming out as
mildly *harmful*; this run has it mildly helpful. When the sign of an effect
flips on a rebuild, the effect is noise, and no amount of narrating it will
make it a finding.

**The Oscar result is the interesting one**, because the idea was reasonable
and it fails in a specific way rather than a boring one. Forty percent of this
sample has an Academy Award winner attached, so the feature is neither rare nor
thinly covered. Prestige is simply a different axis from commercial
performance, and what the model needs to know about a film's scale it already
has from budget and distributor.

That completes a pattern the ablation has now found across eleven feature
groups, without a single exception:

> **What the film is predicts revenue. Who made it does not.**

The 35-feature model ships. Thirteen features that buy nothing are thirteen
features to maintain, refresh and explain, and the code stays in the repository
as the record of the test rather than as part of the product.

---
## Decision 10: a claim that did not survive contact

The original design argued that worldwide should be the **sum** of a domestic
fit and an international fit, because the two halves have different drivers.
That is a reasonable-sounding assertion, so it was tested rather than left
standing.

It loses. Each half is fit in log space and exponentiated before summing, so
two independent errors compound instead of cancelling, while a direct fit
optimises the quantity actually wanted. The README was rewritten to say so.

What survived is the other half of the argument: international is the harder
side by roughly ten points.


In [14]:
from boxoffice.model.splits import run as run_split

split = run_split()
print(f"{split.attrs['counts']['usable']} films with both figures")
display(split)
split[['domestic','international','ww_from_split','ww_direct']].mean().to_frame('mean')

686 films with both figures


,year,n,domestic,international,ww_from_split,ww_direct
0,2012,30,0.733333,0.700000,0.833333,0.833333
1,2013,33,0.757576,0.696970,0.696970,0.666667
2,2014,35,0.742857,0.685714,0.742857,0.714286
3,2015,36,0.722222,0.583333,0.694444,0.750000
4,2016,28,0.642857,0.571429,0.678571,0.678571
5,2017,40,0.700000,0.600000,0.675000,0.675000
6,2018,35,0.742857,0.600000,0.742857,0.742857
7,2021,20,0.550000,0.500000,0.450000,0.600000
8,2022,34,0.764706,0.588235,0.735294,0.705882


,mean
domestic,0.706268
international,0.613965
ww_from_split,0.694370
ww_direct,0.707400


---
## Decision 11: what a projection may be shown next to

Fitting one model on everything and printing its prediction beside the real
gross produces a beautiful scatter and grades the model's own memory.

Every released film is instead scored by the model from the fold where its own
year was the test year. The number beside a 2019 film is what a forecaster
standing in December 2018 would have said. Films released before the first
validation fold get **no** projection rather than an in-sample one.


In [15]:
proj = pd.read_parquet(ROOT / 'backend/boxoffice/data/projections.parquet')
scored = proj.dropna(subset=['projected_worldwide'])
print(f'{len(proj):,} films, {len(scored):,} with an out-of-sample projection')
print(f"within a factor of two: {scored.within_2x.mean():.1%}")
print(f"no projection offered:  {proj.projected_worldwide.isna().sum():,}")

show = scored.assign(
    actual=lambda d: (d.actual_worldwide/1e6).round(0),
    projected=lambda d: (d.projected_worldwide/1e6).round(0))
show.nlargest(8, 'actual_worldwide')[['title','actual','projected','ratio']]

2,577 films, 1,498 with an out-of-sample projection
within a factor of two: 52.7%
no projection offered:  1,079


,title,actual,projected,ratio
1676,Star Wars: The Force Awakens,2068.0,741.0,0.358426
1933,Avengers: Infinity War,2052.0,1299.0,0.633046
2166,Spider-Man: No Way Home,1921.0,707.0,0.368032
2346,Inside Out 2,1699.0,661.0,0.389283
1615,Jurassic World,1672.0,513.0,0.307035
1317,The Avengers,1519.0,685.0,0.451036
1597,Furious 7,1515.0,761.0,0.502104
2197,Top Gun: Maverick,1489.0,564.0,0.379074


---
## What is not established

* **2020 breaks.** Both model and baseline collapse. The pandemic severed the
  budget-to-gross relationship and a model trained through 2019 had no way to
  know. That is temporal validation being honest, not a bug.
* **Domestic coverage is 28%** and thinnest in the recent years that matter
  most, so the three-way split above is directional. A daily job is filling it.
* **The sample is studio films.** It will not price a microbudget breakout,
  because the frame excluded films no distributor picked up.
* **No learner is meaningfully better than any other here.** Ridge, a random
  forest and two boosters land within two points of each other, so the error
  that remains is a property of what is knowable before release rather than of
  the estimator. More modelling will not move it; better features, or an
  admission that some of opening weekend is unforecastable, might.
* **Cast contributing nothing may be a sample-size result** rather than a truth
  about stardom. 2,457 films is not many, and prior gross is a crude proxy for
  drawing power.
